# 08 Binaries and Image Reconstruction

**Series:** Physics of the CHARA Array (Mount Wilson Observatory)
**Topic T8:** turning $V^2$ and closure phases into science: finding binary companions and making images.

The previous notebooks built up what CHARA measures:

- $V^2$, the squared fringe contrast, on each baseline (notebooks 03 and 04).
- Closure phases, on each triangle of telescopes (notebook 07).
- Both over many baselines and hours, the uv coverage (notebook 05).

This notebook uses them in two ways:

1. **Model fitting.** If you know roughly what the source is, such as a binary star, fit its few parameters directly.
2. **Image reconstruction.** If you don't, solve for a whole picture that fits the data, with sensible extra assumptions to fill the gaps in the uv coverage.

It ends with an image of the red giant CL Lac, reconstructed here from real MIRC-X data.

Images in this notebook follow the astronomers' convention: **north up, east to the left**, as the sky looks from below.

In [ ]:
%matplotlib inline
import csv
import io
import itertools
import time
import urllib.parse
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy.optimize import minimize
from scipy.stats import chi2 as chi2_distribution, norm
from ipywidgets import interact, FloatSlider, FloatLogSlider, Dropdown, Checkbox

RAD_TO_MAS = 180 / np.pi * 3600 * 1000
MAS_TO_RAD = 1 / RAD_TO_MAS

BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#8a8a85"
SLIDER_STYLE = {"handle_color": BLUE}  # visible knob on white backgrounds
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3df",
    "lines.linewidth": 2,
})

STATIONS = {
    "S1": (0.000, 0.000, 0.000),
    "S2": (-5.748, 33.581, 0.637),
    "E1": (125.333, 305.928, -5.919),
    "E2": (70.389, 269.715, -2.803),
    "W1": (-175.068, 216.327, -10.798),
    "W2": (-69.085, 199.342, 0.471),
}
NAMES = list(STATIONS)
PAIRS = list(itertools.combinations(range(6), 2))
TRIANGLES = list(itertools.combinations(range(6), 3))
LATITUDE = np.radians(34.2244)


def project(enu, hour_angle, dec):
    # Projected baseline (u, v) in metres, as in notebook 05
    e, n, up = enu
    x = -np.sin(LATITUDE) * n + np.cos(LATITUDE) * up
    z = np.cos(LATITUDE) * n + np.sin(LATITUDE) * up
    u = np.sin(hour_angle) * x + np.cos(hour_angle) * e
    v = (-np.sin(dec) * np.cos(hour_angle) * x + np.sin(dec) * np.sin(hour_angle) * e
         + np.cos(dec) * z)
    return u, v


def wrap_rad(phase):
    return (phase + np.pi) % (2 * np.pi) - np.pi


def simulated_coverage(dec_deg=40, hours=6, snapshots=12, wavelengths_um=(1.55, 1.65, 1.75)):
    # uv points (in wavelengths) for all 15 baselines, and the 20 triangles as (uv index, sign) legs
    u, v, legs = [], [], []
    for ha_h in np.linspace(-hours / 2, hours / 2, snapshots):
        for wl in wavelengths_um:
            index = {}
            for i, j in PAIRS:
                um, vm = project(np.subtract(STATIONS[NAMES[j]], STATIONS[NAMES[i]]),
                                 np.radians(15 * ha_h), np.radians(dec_deg))
                index[(i, j)] = len(u)
                u.append(um / (wl * 1e-6))
                v.append(vm / (wl * 1e-6))
            for i, j, k in TRIANGLES:
                legs.append([(index[(i, j)], 1), (index[(j, k)], 1), (index[(i, k)], -1)])
    return np.array(u), np.array(v), np.array(legs)


def closure_phases(vis, legs):
    return np.sum(legs[..., 1] * np.angle(vis[..., legs[..., 0]]), axis=-1)


U_SIM, V_SIM, LEGS_SIM = simulated_coverage()
print(f"Simulated observation: {len(U_SIM)} V² points and {len(LEGS_SIM)} closure phases "
      f"(6 telescopes, 12 snapshots over 6 hours, 3 wavelengths)")

## 1. What a binary looks like to an interferometer

Two point stars, a primary and a fainter companion. With flux ratio $f$ and the companion at offset $(x, y)$, the visibility is

$$V(u, v) = \frac{1 + f\, e^{-2\pi i (u x + v y)}}{1 + f}$$

- $V^2$ is a set of **stripes** across the uv plane, perpendicular to the line joining the stars. The stripe spacing is 1/separation. Wider binaries give tighter stripes.
- The contrast of the stripes depends on $f$. An equal binary swings from 1 down to 0. A faint companion makes only a small ripple.
- The **closure phase** is zero for an equal binary (it is symmetric) and non-zero otherwise.

The left panel shows $V^2$ over the uv plane with CHARA's tracks for a star at declination 40° over 6 hours (white). The right panel shows the closure phase on three triangles through the night.

In [ ]:
def binary_visibility(u, v, x_mas, y_mas, ratio):
    return (1 + ratio * np.exp(-2j * np.pi * (u * x_mas + v * y_mas) * MAS_TO_RAD)) / (1 + ratio)


SHOWN_TRIANGLES = [("S1", "E1", "W1"), ("S1", "S2", "E1"), ("E2", "W1", "W2")]


@interact(
    sep_mas=FloatSlider(value=1.5, min=0.3, max=5, step=0.1, description="sep (mas)",
                        continuous_update=False, style=SLIDER_STYLE),
    pa_deg=FloatSlider(value=60, min=0, max=180, step=5, description="PA (°)",
                       continuous_update=False, style=SLIDER_STYLE),
    ratio=FloatSlider(value=0.4, min=0.02, max=1, step=0.02, description="flux ratio",
                      continuous_update=False, style=SLIDER_STYLE),
)
def show_binary_signature(sep_mas, pa_deg, ratio):
    # Position angle is measured from north towards east
    x, y = sep_mas * np.sin(np.radians(pa_deg)), sep_mas * np.cos(np.radians(pa_deg))
    grid_m = np.linspace(-340, 340, 241)
    gu, gv = np.meshgrid(grid_m / 1.65e-6, grid_m / 1.65e-6)
    v2_map = np.abs(binary_visibility(gu, gv, x, y, ratio)) ** 2

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 5), gridspec_kw={"width_ratios": [1, 1.25]})
    im = ax1.imshow(v2_map, extent=[-340, 340, -340, 340], origin="lower", cmap="viridis",
                    vmin=0, vmax=1)
    fig.colorbar(im, ax=ax1, fraction=0.046, label="V²")
    ha = np.radians(15 * np.linspace(-3, 3, 100))
    for i, j in PAIRS:
        u, v = project(np.subtract(STATIONS[NAMES[j]], STATIONS[NAMES[i]]), ha, np.radians(40))
        ax1.plot(u, v, color="white", lw=1)
        ax1.plot(-u, -v, color="white", lw=1)
    ax1.set_xlabel("u (m)")
    ax1.set_ylabel("v (m)")
    ax1.set_title("V² of the binary across the uv plane, at 1.65 µm", fontsize=10)
    ax1.grid(False)

    hours = np.linspace(-3, 3, 200)
    for names, colour in zip(SHOWN_TRIANGLES, [BLUE, ORANGE, AQUA]):
        total = 0
        for a, b in [(names[0], names[1]), (names[1], names[2]), (names[2], names[0])]:
            u, v = project(np.subtract(STATIONS[b], STATIONS[a]), np.radians(15 * hours), np.radians(40))
            total = total + np.angle(binary_visibility(u / 1.65e-6, v / 1.65e-6, x, y, ratio))
        ax2.plot(hours, np.degrees(wrap_rad(total)), ".", ms=3, color=colour, label="-".join(names))
    ax2.set_ylim(-185, 185)
    ax2.set_yticks([-180, -90, 0, 90, 180])
    ax2.set_xlabel("Hour angle (h)")
    ax2.set_ylabel("Closure phase (°)")
    ax2.set_title("Closure phase through the night", fontsize=10)
    ax2.legend(fontsize=8, loc="lower right", markerscale=4)
    plt.tight_layout()
    plt.show()

**Try this:**

- Raise the separation. The stripes get closer together and the tracks cross many of them.
- Rotate the position angle. The stripes turn with the line joining the stars.
- Set the flux ratio to 1. The stripes reach $V^2 = 0$, and every closure phase is 0° or ±180°.
- Lower the flux ratio to 0.02. The $V^2$ ripples shrink to a few percent, and the closure phases wiggle by only a degree or two. Both signals are small, so faint companions need precise data. Section 2 shows which signal matters more.

## 2. Finding a faint companion

To search for a companion, try every possible position on a grid. At each position find the best flux ratio, and record how well the binary model fits ($\chi^2$). The result is a **$\chi^2$ map**: dark where a companion fits the data well.

The improvement over a single star, $\Delta\chi^2$, says whether the companion is real. The binary model adds three parameters (two for position, one for flux ratio), and the standard $\chi^2$ test turns $\Delta\chi^2$ into a significance in sigmas.

The simulated data below contain a companion 1.2 mas from the primary, towards the north-east, with the noise levels set by the sliders. The defaults reflect real instruments: $V^2$ is limited by calibration to a few percent, while closure phases, which need no calibration star to cancel the atmosphere, reach a fraction of a degree. Untick the closure phases to see what $V^2$ alone can and cannot tell.

In [ ]:
TRUE_COMPANION = (1.04, 0.60)  # mas east, mas north
SEARCH_AXIS = np.linspace(-2.5, 2.5, 61)
RATIO_GRID = np.geomspace(0.002, 1, 16)


def chi2_map(v2, v2_err, cp, cp_err, use_cp):
    sx, sy = np.meshgrid(SEARCH_AXIS, SEARCH_AXIS)
    shift = np.exp(-2j * np.pi * (np.outer(sx.ravel(), U_SIM) + np.outer(sy.ravel(), V_SIM)) * MAS_TO_RAD)
    best = np.full(sx.size, np.inf)
    best_ratio = np.zeros(sx.size)
    for ratio in RATIO_GRID:
        vis = (1 + ratio * shift) / (1 + ratio)
        chi2 = np.sum(((np.abs(vis) ** 2 - v2) / v2_err) ** 2, axis=1)
        if use_cp:
            chi2 += np.sum((wrap_rad(closure_phases(vis, LEGS_SIM) - cp) / cp_err) ** 2, axis=1)
        better = chi2 < best
        best[better], best_ratio[better] = chi2[better], ratio
    single = np.sum(((1 - v2) / v2_err) ** 2) + (np.sum((wrap_rad(cp) / cp_err) ** 2) if use_cp else 0)
    return best.reshape(sx.shape), best_ratio.reshape(sx.shape), single


@interact(
    ratio=FloatLogSlider(value=0.01, base=10, min=-3, max=0, step=0.1, description="flux ratio",
                         continuous_update=False, style=SLIDER_STYLE),
    v2_noise=FloatSlider(value=0.05, min=0.005, max=0.1, step=0.005, description="V² error",
                         continuous_update=False, style=SLIDER_STYLE),
    cp_noise_deg=FloatSlider(value=0.5, min=0.2, max=10, step=0.2, description="CP error (°)",
                             continuous_update=False, style=SLIDER_STYLE),
    use_cp=Checkbox(value=True, description="use closure phases"),
)
def show_detection(ratio, v2_noise, cp_noise_deg, use_cp):
    rng = np.random.default_rng(5)
    vis = binary_visibility(U_SIM, V_SIM, *TRUE_COMPANION, ratio)
    cp_err = np.radians(cp_noise_deg)
    v2 = np.abs(vis) ** 2 + rng.normal(0, v2_noise, len(vis))
    cp = closure_phases(vis, LEGS_SIM) + rng.normal(0, cp_err, len(LEGS_SIM))

    chi2, ratios, single = chi2_map(v2, v2_noise, cp, cp_err, use_cp)
    iy, ix = np.unravel_index(np.argmin(chi2), chi2.shape)
    delta = single - chi2.min()
    sigma = norm.isf(chi2_distribution.sf(delta, 3) / 2) if delta > 0 else 0.0

    fig, ax = plt.subplots(figsize=(6.4, 5.6))
    extent = [SEARCH_AXIS[0], SEARCH_AXIS[-1]] * 2
    im = ax.imshow(chi2 - chi2.min(), extent=extent, origin="lower", cmap="magma",
                   vmax=max(4 * np.percentile(chi2 - chi2.min(), 5), 25))
    fig.colorbar(im, ax=ax, fraction=0.046, label="χ² above the best fit")
    ax.scatter(0, 0, marker="*", s=200, color="white", label="primary")
    ax.scatter(*TRUE_COMPANION, s=160, facecolor="none", edgecolor=AQUA, lw=2, label="true companion")
    ax.scatter(SEARCH_AXIS[ix], SEARCH_AXIS[iy], marker="x", s=100, color=BLUE, lw=2.5, label="best fit")
    ax.invert_xaxis()
    ax.set_xlabel("East offset (mas), east is left")
    ax.set_ylabel("North offset (mas)")
    ax.set_title("χ² map of companion position" + ("" if use_cp else " (V² only)"), fontsize=10)
    ax.legend(fontsize=8, loc="lower left")
    ax.grid(False)
    plt.tight_layout()
    plt.show()
    print(f"Best fit: {SEARCH_AXIS[ix]:+.2f} mas east, {SEARCH_AXIS[iy]:+.2f} mas north, "
          f"flux ratio {ratios[iy, ix]:.3f} (true {ratio:.3f})")
    significance = f"about {sigma:.1f} sigma" if sigma < 8 else "more than 8 sigma"
    print(f"Improvement over a single star: Δχ² = {delta:.0f}, {significance}")

**Try this:**

- At the defaults, a companion 1% as bright as the primary is found at its true position with $\Delta\chi^2$ of about 1000.
- Untick the closure phases. $\Delta\chi^2$ drops about tenfold, and a second, equally good minimum appears on the opposite side of the primary. $V^2$ is the same for an image and its 180° rotation, so only the closure phase tells which side the companion is on.
- Lower the flux ratio to 0.003. With closure phases the companion is still found. Without them it is a marginal detection, on the wrong side.
- Lower it to 0.001. The minimum wanders off and the significance falls to about 2 sigma: no detection.
- Back at 1%, raise the closure-phase error to 5°. $\Delta\chi^2$ drops from about 1000 to about 130.

CHARA uses searches like this to find close companions and to follow binary orbits, which give stellar masses.

## 3. From data to images

In notebook 05 a **dirty image** was made by adding up the Fourier components. That needs the full complex visibility on each baseline, amplitude **and** phase. The atmosphere removed the phases. What remains is $V^2$ and closure phases, so the image cannot be computed directly.

Image reconstruction turns the problem around. Start with a guess of the image, compute the $V^2$ and closure phases it would produce, and adjust the pixels until they match the data. Because the uv coverage has gaps, many images fit the data equally well. To pick one, add a **regularization** term that prefers simple images:

$$\text{cost} = \chi^2_{V^2} + \chi^2_\text{closure} + \mu \times \text{(distance from a smooth prior image)}$$

Here the distance is the **maximum entropy** measure, used by programs such as BSMEM. With no other evidence it keeps the image close to a smooth blob, and it keeps every pixel positive. The weight $\mu$ sets the balance:

- Small $\mu$ trusts the data, at the risk of noisy, speckled detail.
- Large $\mu$ gives a smooth image that may no longer fit the data.

A good reconstruction has a reduced $\chi^2$ near 1. That means the image matches the data to within the error bars, and no better.

The cell below defines the solver. It computes the gradient of the cost exactly, so scipy's L-BFGS-B optimizer can adjust over a thousand pixels in a few seconds.

The cost has many local minima, because $V^2$ and closure phases depend on the image in a non-linear way. The solver starts from a plain disk. If it gets stuck (reduced $\chi^2$ above 5), it tries once more with the pixels written as exponentials, which changes the path the optimizer takes, and keeps the better fit.

In [ ]:
def dft_matrices(u, v, axis_mas):
    # exp(-2πi(ux + vy)) splits into an x part and a y part (x = east, y = north)
    ex = np.exp(-2j * np.pi * np.outer(u, axis_mas * MAS_TO_RAD))
    ey = np.exp(-2j * np.pi * np.outer(v, axis_mas * MAS_TO_RAD))
    return ex, ey


def image_visibility(image, ex, ey):
    return np.einsum("ky,kx,yx->k", ey, ex, image, optimize=True) / image.sum()


def cost_and_gradient(x, data, ex, ey, prior, mu):
    n = int(np.sqrt(x.size))
    image = x.reshape(n, n)
    flux = image.sum()
    vis = np.einsum("ky,kx,yx->k", ey, ex, image, optimize=True) / flux

    # grad_vis[k] collects d(chi2)/dV as a complex number: d(chi2) = Re(conj(grad_vis) dV)
    grad_vis = np.zeros(len(vis), complex)
    ok = np.isfinite(data["v2"])
    resid = (np.abs(vis[ok]) ** 2 - data["v2"][ok]) / data["v2_err"][ok]
    chi2 = np.sum(resid ** 2)
    grad_vis[ok] = 4 * resid / data["v2_err"][ok] * vis[ok]

    idx, sign = data["legs"][..., 0], data["legs"][..., 1]
    diff = np.sum(sign * np.angle(vis[idx]), axis=1) - data["cp"]
    chi2 += np.sum(2 * (1 - np.cos(diff)) / data["cp_err"] ** 2)  # wraps safely, ≈ diff² for small diff
    weight = 2 * np.sin(diff) / data["cp_err"] ** 2
    for leg in range(3):
        np.add.at(grad_vis, idx[:, leg], weight * sign[:, leg] * 1j / np.conj(vis[idx[:, leg]]))

    grad_image = (np.real(np.einsum("k,ky,kx->yx", np.conj(grad_vis), ey, ex, optimize=True))
                  - np.real(np.sum(np.conj(grad_vis) * vis))) / flux
    entropy = np.sum(x * np.log(x / prior) - x + prior)
    n_data = ok.sum() + len(diff)
    cost = (chi2 + mu * n_data * entropy) / n_data
    gradient = (grad_image.ravel() + mu * n_data * np.log(x / prior)) / n_data
    return cost, gradient


def log_pixel_cost(z, *args):
    # Same cost with pixels written as exp(z): always positive, no bounds needed
    x = np.exp(z)
    cost, gradient = cost_and_gradient(x, *args)
    return cost, gradient * x


def reconstruct(data, axis_mas, prior_image, mu, start, iterations=500):
    ex, ey = dft_matrices(data["u"], data["v"], axis_mas)
    prior = prior_image.ravel() / prior_image.sum()
    x0 = start.ravel() / start.sum()
    args = (data, ex, ey, prior, mu)
    options = {"maxiter": iterations, "ftol": 1e-12, "gtol": 1e-10}
    x = minimize(cost_and_gradient, x0, args=args, jac=True, method="L-BFGS-B",
                 bounds=[(1e-14, None)] * x0.size, options=options).x
    reduced_chi2 = cost_and_gradient(x, data, ex, ey, prior, 0.0)[0]
    if reduced_chi2 > 5:
        # Stuck in a poor local minimum: try again with log pixels and keep the better fit
        z = minimize(log_pixel_cost, np.log(x0), args=args, jac=True, method="L-BFGS-B",
                     options=options).x
        retry = cost_and_gradient(np.exp(z), data, ex, ey, prior, 0.0)[0]
        if retry < reduced_chi2:
            x, reduced_chi2 = np.exp(z), retry
    image = x.reshape(prior_image.shape)
    return image / image.max(), reduced_chi2


def show_sky(ax, image, axis_mas, title, cmap="inferno"):
    half = (axis_mas[1] - axis_mas[0]) / 2
    extent = [axis_mas[0] - half, axis_mas[-1] + half, axis_mas[0] - half, axis_mas[-1] + half]
    ax.imshow(image, extent=extent, origin="lower", cmap=cmap)
    ax.invert_xaxis()
    ax.set_xlabel("East (mas), east is left")
    ax.set_title(title, fontsize=10)
    ax.grid(False)

### Testing on skies we know

Three test skies, observed with the same simulated coverage as section 2 (6 telescopes, 6 hours, 3 wavelengths). The noise is 3% on $V^2$ plus a small floor. Closure-phase errors start at 2° and grow where the visibility is low, as in real data. The prior is a round Gaussian blob.

The middle panel is the dirty image computed **with** the true phases, which a real observation never has. It shows what the coverage alone can do. The right panel uses only $V^2$ and closure phases.

Each reconstruction takes a few seconds.

In [ ]:
SIM_AXIS = (np.arange(40) - 19.5) * 0.15   # 40 × 40 pixels of 0.15 mas
SX, SY = np.meshgrid(SIM_AXIS, SIM_AXIS)


def blob(x0, y0, width, flux=1.0):
    return flux * np.exp(-((SX - x0) ** 2 + (SY - y0) ** 2) / (2 * width ** 2))


def spotted_star():
    r = np.hypot(SX, SY) / 1.5
    disk = np.where(r < 1, 0.4 + 0.6 * np.sqrt(np.clip(1 - r ** 2, 0, 1)), 0)
    return disk * (1 + blob(-0.6, 0.5, 0.35, 1.2) - blob(0.5, -0.4, 0.3, 0.5))


def oval_rotator():
    # Elongated, fast-spinning star, brighter at the poles (gravity darkening), dimmer at the edge,
    # long axis tilted 30°
    t = np.radians(30)
    along = SX * np.cos(t) - SY * np.sin(t)
    across = SX * np.sin(t) + SY * np.cos(t)
    r = np.hypot(along / 1.7, across / 1.35)
    limb = np.clip(1 - r ** 2, 0, 1) ** 0.3
    return np.where(r < 1, (1 - 0.55 * (along / 1.7) ** 2) * limb, 0)


TEST_SKIES = {
    "Binary, 2:1, 1.6 mas apart": lambda: blob(0.6, -0.5, 0.1) + blob(-0.4, 0.75, 0.1, 0.5),
    "Spotted star, 3 mas": spotted_star,
    "Oval fast rotator (like Altair)": oval_rotator,
}
PRIOR_SIM = blob(0, 0, 1.2)
START_SIM = (np.hypot(SX, SY) < 1.5) + 0.01   # a plain 3 mas disk to start from


def simulate_data(truth, rng):
    ex, ey = dft_matrices(U_SIM, V_SIM, SIM_AXIS)
    vis = image_visibility(truth, ex, ey)
    v2_err = 0.03 * np.abs(vis) ** 2 + 0.002
    # Closure phases are noisier where the visibility is low, as in real data
    leg_err = 0.005 / np.maximum(np.abs(vis[LEGS_SIM[..., 0]]), 1e-3)
    cp_err = np.clip(np.sqrt(np.sum(leg_err ** 2, axis=1)), np.radians(2), np.radians(60))
    data = dict(u=U_SIM, v=V_SIM, legs=LEGS_SIM,
                v2=np.abs(vis) ** 2 + rng.normal(0, 1, len(vis)) * v2_err, v2_err=v2_err,
                cp=closure_phases(vis, LEGS_SIM) + rng.normal(0, 1, len(LEGS_SIM)) * cp_err,
                cp_err=cp_err)
    dirty = np.real(np.einsum("k,ky,kx->yx", vis, np.conj(ey), np.conj(ex), optimize=True))
    return data, dirty


@interact(
    sky=Dropdown(options=list(TEST_SKIES), value="Spotted star, 3 mas", description="Sky",
                 layout={"width": "360px"}),
    mu=Dropdown(options=[0.01, 0.1, 1.0, 10.0, 100.0], value=1.0, description="μ"),
)
def show_reconstruction(sky, mu):
    truth = TEST_SKIES[sky]()
    data, dirty = simulate_data(truth, np.random.default_rng(6))
    start = time.time()
    image, reduced_chi2 = reconstruct(data, SIM_AXIS, PRIOR_SIM, mu, start=START_SIM)

    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.6))
    show_sky(axes[0], truth, SIM_AXIS, "True sky")
    show_sky(axes[1], dirty, SIM_AXIS, "Dirty image, if phases were known", cmap="RdBu_r")
    show_sky(axes[2], image, SIM_AXIS, f"Reconstruction from V² and closure phase\n"
                                       f"μ = {mu}, reduced χ² = {reduced_chi2:.2f}")
    axes[0].set_ylabel("North (mas)")
    plt.tight_layout()
    plt.show()
    print(f"Reconstruction took {time.time() - start:.1f} s")

**Try this:**

- The spotted star at $\mu = 1$: the bright spot comes back in the right place, and the dark spot faintly, from data with no baseline phases at all. The reduced $\chi^2$ is about 1.
- Lower $\mu$ to 0.1 or 0.01. The reduced $\chi^2$ dips just below 1 and the disk fills with speckle. The image is fitting the noise.
- Raise $\mu$ to 100. The image is smoother and closer to the prior blob, and the reduced $\chi^2$ rises well above 1: it no longer fits the data.
- The binary: both stars are recovered with the right separation and brightness. The pair may sit slightly off-centre, because $V^2$ and closure phases do not change when the whole image shifts. Only relative positions are measured.
- The oval rotator is harder. The reconstruction gets the size right but stays nearly round and invents structure. Its smooth edges and soft pole-to-equator change are near the limit of this coverage and this simple solver. CHARA's images of Altair (Monnier et al. 2007, Science 317, 342) used more data and were checked against physical models of a spinning star. For a source like this, fitting a model is often the better tool.

## 4. A real image: the surface of CL Lac

Notebook 07 showed that CL Lac's closure phases are far from symmetric. Now reconstruct its image from the real MIRC-X data, one spectral channel at a time. The star's apparent size changes across the band, because the atmosphere of a red giant is more opaque at some wavelengths, so mixing channels would blur the result.

Chiavassa et al. (2020, A&A 640, A23) reconstructed these same data with the MiRA software. They found a disk about 3 mas across with a **brighter area in the south-east**, which they compared with simulations of giant convection cells on the star's surface. They reached a reduced $\chi^2$ of about 2.

The solver here is simpler than MiRA, with a different regularization. If it finds the same features, that is a fair check that they come from the data and not from the method.

One data detail: a few measurements have most of their spectral channels flagged by the reduction pipeline, and the rest are unreliable too. One reports $V^2 = 0.003$ on the shortest baseline, where all its neighbours give about 0.2. Measurements with fewer than 4 usable channels are skipped.

In [ ]:
TAP_URL = "https://tap.jmmc.fr/vollt/tap/sync"
OIDB_URL = "https://oidb.jmmc.fr"
CL_LAC_DIR = Path("data/irc_50448")


def download_cl_lac():
    # Public MIRC-X data on CL Lac (IRC+50448) from the JMMC Optical Interferometry Database
    done_marker = CL_LAC_DIR / ".complete"
    if done_marker.exists():
        return sorted(CL_LAC_DIR.glob("*.fits"))
    CL_LAC_DIR.mkdir(parents=True, exist_ok=True)
    query = ("SELECT access_url FROM oidb WHERE facility_name = 'CHARA' AND data_rights = 'public' "
             "AND calib_level = 3 AND target_name = 'IRC+50448'")
    params = urllib.parse.urlencode({"REQUEST": "doQuery", "LANG": "ADQL",
                                     "FORMAT": "csv", "QUERY": query}).encode()
    with urllib.request.urlopen(TAP_URL, data=params, timeout=60) as response:
        rows = list(csv.DictReader(io.StringIO(response.read().decode())))

    def fetch(row):
        url = urllib.parse.urljoin(OIDB_URL, row["access_url"])
        target = CL_LAC_DIR / Path(urllib.parse.urlparse(url).path).name
        if not target.exists():
            urllib.request.urlretrieve(url, target)

    with ThreadPoolExecutor(max_workers=8) as pool:
        list(pool.map(fetch, rows))
    done_marker.touch()
    return sorted(CL_LAC_DIR.glob("*.fits"))


MIN_USABLE_CHANNELS = 4


def load_channel(files, channel):
    # V² and closure phases for one spectral channel. Each triangle leg points at a V² entry.
    u, v, v2, v2_err, cp, cp_err, legs = [], [], [], [], [], [], []
    for path in files:
        with fits.open(path) as hdul:
            wavelength = hdul["OI_WAVELENGTH"].data["EFF_WAVE"][channel]
            index = {}
            for row in hdul["OI_VIS2"].data:
                index[tuple(int(i) for i in row["STA_INDEX"])] = len(u)
                u.append(row["UCOORD"] / wavelength)
                v.append(row["VCOORD"] / wavelength)
                usable = ~row["FLAG"] & (row["VIS2ERR"] > 0)
                # Rows with most channels flagged are unreliable even where unflagged
                good = usable[channel] and usable.sum() >= MIN_USABLE_CHANNELS
                v2.append(row["VIS2DATA"][channel] if good else np.nan)
                v2_err.append(row["VIS2ERR"][channel] if good else np.nan)
            for row in hdul["OI_T3"].data:
                usable = ~row["FLAG"] & (row["T3PHIERR"] > 0)
                if not usable[channel] or usable.sum() < MIN_USABLE_CHANNELS:
                    continue
                a, b, c = (int(i) for i in row["STA_INDEX"])
                legs.append([(index[(p, q)], 1) if (p, q) in index else (index[(q, p)], -1)
                             for p, q in ((a, b), (b, c), (c, a))])
                cp.append(row["T3PHI"][channel])
                cp_err.append(row["T3PHIERR"][channel])
    return dict(u=np.array(u), v=np.array(v), v2=np.array(v2), v2_err=np.array(v2_err),
                cp=np.radians(cp), cp_err=np.radians(cp_err), legs=np.array(legs))


CL_LAC_FILES = download_cl_lac()
CHANNEL_WAVELENGTHS = fits.getdata(CL_LAC_FILES[0], "OI_WAVELENGTH")["EFF_WAVE"] * 1e6
# Channels 0 and 7, at the band edges, are flagged in every file
CHANNELS = {f"{CHANNEL_WAVELENGTHS[c]:.3f} µm": c for c in range(1, 7)}
REAL_AXIS = (np.arange(40) - 19.5) * 0.2   # 40 × 40 pixels of 0.2 mas
RX, RY = np.meshgrid(REAL_AXIS, REAL_AXIS)
PRIOR_REAL = np.exp(-(RX ** 2 + RY ** 2) / (2 * 1.5 ** 2))
START_REAL = (np.hypot(RX, RY) < 1.3) + 0.01   # a plain 2.6 mas disk to start from
print("Channels:", ", ".join(CHANNELS))

In [ ]:
@interact(
    channel=Dropdown(options=list(CHANNELS), value=list(CHANNELS)[0], description="Channel"),
    mu=Dropdown(options=[0.01, 0.1, 1.0], value=0.1, description="μ"),
)
def show_cl_lac(channel, mu):
    data = load_channel(CL_LAC_FILES, CHANNELS[channel])
    start = time.time()
    image, reduced_chi2 = reconstruct(data, REAL_AXIS, PRIOR_REAL, mu, start=START_REAL)

    ex, ey = dft_matrices(data["u"], data["v"], REAL_AXIS)
    model = image_visibility(image, ex, ey)
    ok = np.isfinite(data["v2"])
    sf = np.hypot(data["u"], data["v"]) / 1e6
    longest = sf[data["legs"][..., 0]].max(axis=1)

    fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.6))
    show_sky(axes[0], image, REAL_AXIS, f"CL Lac at {channel}, μ = {mu}\nreduced χ² = {reduced_chi2:.2f}")
    axes[0].set_xlim(3, -3)
    axes[0].set_ylim(-3, 3)
    axes[0].set_ylabel("North (mas)")

    axes[1].errorbar(sf[ok], data["v2"][ok], yerr=data["v2_err"][ok], fmt="o", ms=2, color=GREY,
                     alpha=0.6, label="data")
    axes[1].scatter(sf[ok], np.abs(model[ok]) ** 2, s=4, color=ORANGE, label="image")
    axes[1].set_yscale("log")
    axes[1].set_ylim(1e-4, 1.2)
    axes[1].set_xlabel("Spatial frequency (millions of wavelengths)")
    axes[1].set_ylabel("V² (log scale)")
    axes[1].set_title("V²: data and reconstructed image", fontsize=10)
    axes[1].legend(fontsize=8)

    axes[2].errorbar(longest, np.degrees(wrap_rad(data["cp"])), yerr=np.degrees(data["cp_err"]),
                     fmt="o", ms=2, color=GREY, alpha=0.5, label="data")
    axes[2].scatter(longest, np.degrees(wrap_rad(closure_phases(model, data["legs"]))), s=4,
                    color=ORANGE, label="image")
    axes[2].set_ylim(-190, 190)
    axes[2].set_yticks([-180, -90, 0, 90, 180])
    axes[2].set_xlabel("Longest baseline in the triangle (millions of wavelengths)")
    axes[2].set_ylabel("Closure phase (°)")
    axes[2].set_title("Closure phase: data and reconstructed image", fontsize=10)
    axes[2].legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    print(f"Reconstruction took {time.time() - start:.1f} s, using {ok.sum()} V² and {len(data['cp'])} closure phases")
    print(f"Disk outline (pixels above 20% of the peak) about "
          f"{2 * np.sqrt((image > 0.2).sum() / np.pi) * 0.2:.1f} mas across")

**What to look for:**

- A resolved disk about 2.5 mas across, with its brightest region south of centre and slightly east. It appears at every wavelength. Chiavassa et al. found a brighter area in the south-east with different software and different regularization. Their 3 mas diameter comes from model atmospheres, which include the star's dim outer edge.
- Chiavassa et al. point out that a bright region like this pulls the star's light-weighted centre (photocentre) away from the middle of the disk. For red giants that adds noise to Gaia's measurements of their positions and distances. Measuring that shift needs more care than this simple solver gives, so it is not attempted here.
- The reduced $\chi^2$ is about 2 to 3, not 1, similar to the published value. Real error bars are rarely perfect, and this solver is simpler than MiRA.
- Try other channels and values of $\mu$. Features that survive these changes are more trustworthy than those that don't.

**Caution.** An image reconstruction is one picture consistent with the data, not a photograph. Fine detail near the resolution limit (about 0.5 mas here) depends on the regularization. Published images are always checked by varying the method, as Chiavassa et al. did, and by comparing with physical models.

## Summary

| Idea | What it means |
|---|---|
| Binary signature | Stripes in $V^2$ across the uv plane, non-zero closure phase if unequal |
| $\chi^2$ map | Try every companion position, keep the best flux ratio, look for the minimum |
| Closure phase and side | $V^2$ alone cannot tell an image from its 180° rotation |
| Regularization | Extra preference for simple images, needed because the uv coverage has gaps |
| $\mu$ | Balance between fitting the data and staying close to the prior. Aim for reduced $\chi^2$ near 1 |

1. When the source is simple, fit a model: few parameters, sharp answers.
2. When it is not, reconstruct an image from $V^2$ and closure phases with regularization.
3. Real CHARA data reveal the patchy surface of a red giant 1.2 kpc away, a disk the size of a coin seen from about 1,500 km.

This completes the series. The overview in [00_overview.md](00_overview.md) links all eight topics.